In [1]:
import requests
import json
import pandas as pd
from datetime import datetime
from typing import List, Optional

In [2]:
class GrantsGovAPI:
    """
    Python wrapper for Grants.gov search2 API
    No authentication required for searching grants
    """
    
    def __init__(self, environment='production'):
        """
        Initialize API with environment
        
        Args:
            environment: 'production' or 'staging'
        """
        self.base_urls = {
            'production': 'https://api.grants.gov',
            'staging': 'https://api.staging.grants.gov'
        }
        self.base_url = self.base_urls.get(environment, self.base_urls['production'])
        self.search_endpoint = f"{self.base_url}/v1/api/search2"
        
    def search_grants(self, 
                     keyword=None,
                     agencies=None,
                     funding_categories=None,
                     opp_statuses="posted",
                     eligibilities=None,
                     aln=None,
                     rows=25,
                     start_record=0):
        """
        Search for grants based on filters
        
        Args:
            keyword: Search keywords (e.g., "small business technology")
            agencies: Agency codes (e.g., "HHS", "DOD", "SBA")
            funding_categories: Category codes (e.g., "HL" for Health, "BC" for Business)
            opp_statuses: Status - "posted", "forecasted", "closed", "archived" (can be pipe-separated)
            eligibilities: Eligibility codes (e.g., "13" for small businesses)
            aln: Assistance Listing Number (formerly CFDA)
            rows: Number of results to return (default 25)
            start_record: Starting record for pagination
            
        Returns:
            dict: Response data containing grant opportunities
        """
        
        # Build request body
        payload = {
            "rows": rows,
            "startRecordNum": start_record
        }
        
        # Add optional filters
        if keyword:
            payload["keyword"] = keyword
        if agencies:
            payload["agencies"] = agencies
        if funding_categories:
            payload["fundingCategories"] = funding_categories
        if opp_statuses:
            payload["oppStatuses"] = opp_statuses
        if eligibilities:
            payload["eligibilities"] = eligibilities
        if aln:
            payload["aln"] = aln
            
        # Make API request
        headers = {'Content-Type': 'application/json'}
        
        try:
            response = requests.post(
                self.search_endpoint,
                headers=headers,
                json=payload,
                timeout=30
            )
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.RequestException as e:
            print(f"Error making API request: {e}")
            return None
    
    def search_to_dataframe(self, response_data):
        """
        Convert API response to pandas DataFrame
        
        Args:
            response_data: Response from search_grants()
            
        Returns:
            pandas.DataFrame: Grants data in tabular format
        """
        if not response_data or response_data.get('errorcode') != 0:
            print("Error in API response")
            return pd.DataFrame()
            
        data = response_data.get('data', {})
        opportunities = data.get('oppHits', [])
        
        if not opportunities:
            print("No grants found matching criteria")
            return pd.DataFrame()
        
        # Convert to DataFrame and flatten ALN list
        df = pd.DataFrame(opportunities)
        df['aln'] = df['alnist'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
        df = df.drop('alnist', axis=1)
        
        # Add grant URL
        df['grant_url'] = df['id'].apply(lambda x: f"https://www.grants.gov/search-results-detail/{x}")
        
        # Reorder columns for better readability
        column_order = [
            'number', 'title', 'agencyName', 'agencyCode', 'oppStatus',
            'openDate', 'closeDate', 'aln', 'docType', 'id', 'grant_url'
        ]
        
        # Only include columns that exist
        column_order = [col for col in column_order if col in df.columns]
        df = df[column_order]
        
        return df
    
    def search_business_grants(self,
                              business_keywords: List[str] = None,
                              industry_focus: str = None,
                              research_oriented: bool = False,
                              max_results: int = 100,
                              include_forecasted: bool = True) -> pd.DataFrame:
        """
        Search for small business grants with business-specific parameters
        
        Args:
            business_keywords: List of keywords describing your business activities
                              Examples: ["software development", "medical devices", "renewable energy"]
            industry_focus: Main industry category
                Options: "technology", "health", "energy", "agriculture", "environment",
                        "manufacturing", "education", "defense"
            research_oriented: Whether your business does R&D work (helps find SBIR/STTR grants)
            max_results: Maximum number of grants to return
            include_forecasted: Include upcoming grants (not just currently open)
            
        Returns:
            pandas.DataFrame: All matching grants from SBA, HHS, DOE, DOD, NSF, and other agencies
        """
        
        # Map industry focus to funding categories
        industry_to_category = {
            "technology": "ST",        # Science and Technology
            "health": "HL",            # Health
            "energy": "EN",            # Energy
            "agriculture": "AG",       # Agriculture
            "environment": "EN",       # Environment
            "manufacturing": "BC",     # Business and Commerce
            "education": "ED",         # Education
            "defense": "IS"            # Information and Statistics (closest for defense tech)
        }
        
        # Build keyword search
        search_terms = []
        if business_keywords:
            search_terms.extend(business_keywords)
        if research_oriented:
            search_terms.extend(["SBIR", "STTR", "research", "innovation"])
        
        keyword_string = " ".join(search_terms) if search_terms else None
        
        # Set funding category based on industry
        funding_category = industry_to_category.get(industry_focus) if industry_focus else None
        
        # Set status
        status = "posted|forecasted" if include_forecasted else "posted"
        
        # Multiple agencies - search each and combine results
        agencies_to_search = ["SBA", "HHS", "DOE", "DOD", "NSF", "ED", "USDA", "NASA", "DHS"]
        
        all_grants = []
        
        for agency in agencies_to_search:
            print(f"Searching {agency}...")
            
            response = self.search_grants(
                keyword=keyword_string,
                agencies=agency,
                funding_categories=funding_category,
                opp_statuses=status,
                eligibilities="13",  # Small businesses
                rows=max_results
            )
            
            if response and response.get('errorcode') == 0:
                df = self.search_to_dataframe(response)
                if not df.empty:
                    all_grants.append(df)
        
        # Also do a general search without agency filter to catch others
        print("Searching other agencies...")
        response = self.search_grants(
            keyword=keyword_string,
            funding_categories=funding_category,
            opp_statuses=status,
            eligibilities="13",
            rows=max_results
        )
        
        if response and response.get('errorcode') == 0:
            df = self.search_to_dataframe(response)
            if not df.empty:
                all_grants.append(df)
        
        # Combine all results and remove duplicates
        if all_grants:
            combined_df = pd.concat(all_grants, ignore_index=True)
            combined_df = combined_df.drop_duplicates(subset=['number'], keep='first')
            combined_df = combined_df.reset_index(drop=True)
            
            print(f"\nTotal unique grants found: {len(combined_df)}")
            return combined_df
        else:
            print("No grants found matching your criteria")
            return pd.DataFrame()
    
    def print_results(self, response_data):
        """
        Pretty print search results
        
        Args:
            response_data: Response from search_grants()
        """
        if not response_data or response_data.get('errorcode') != 0:
            print("Error in API response")
            return
            
        data = response_data.get('data', {})
        hit_count = data.get('hitCount', 0)
        opportunities = data.get('oppHits', [])
        
        print(f"\n{'='*80}")
        print(f"Found {hit_count} matching grants")
        print(f"{'='*80}\n")
        
        for i, opp in enumerate(opportunities, 1):
            print(f"{i}. {opp.get('title', 'N/A')}")
            print(f"   Opportunity #: {opp.get('number', 'N/A')}")
            print(f"   Agency: {opp.get('agencyName', 'N/A')} ({opp.get('agencyCode', 'N/A')})")
            print(f"   Status: {opp.get('oppStatus', 'N/A')}")
            print(f"   Open Date: {opp.get('openDate', 'N/A')}")
            print(f"   Close Date: {opp.get('closeDate', 'TBD')}")
            print(f"   ALN: {', '.join(opp.get('alnist', []))}")
            print(f"   URL: https://www.grants.gov/search-results-detail/{opp.get('id', '')}")
            print()



In [4]:
# Example Usage
if __name__ == "__main__":
    # Initialize API
    api = GrantsGovAPI(environment='production')
    
    print("="*80)
    print("BUSINESS GRANT SEARCH - Customized for Your Business")
    print("="*80)
    
    # Define your business parameters
    business_params = {
        "business_keywords": ["software development", "artificial intelligence", "data analytics"],
        "industry_focus": "technology",  # Options: technology, health, energy, agriculture, etc.
        "research_oriented": True,       # Set True if you do R&D
        "max_results": 100,
        "include_forecasted": True       # Include upcoming grants
    }
    
    # Search for grants matching your business
    grants_df = api.search_business_grants(**business_params)
    
    # Display results
    if not grants_df.empty:
        print("\n" + "="*80)
        print(f"Found {len(grants_df)} grants for your business")
        print("="*80 + "\n")
        
        # Show first few rows
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', 50)
        print(grants_df.head(10))
        
        # Show breakdown by agency
        print("\n" + "="*80)
        print("Grants by Agency:")
        print("="*80)
        print(grants_df['agencyName'].value_counts())
        
        # Show breakdown by status
        print("\n" + "="*80)
        print("Grants by Status:")
        print("="*80)
        print(grants_df['oppStatus'].value_counts())
        
        # Export to CSV
        output_filename = f"business_grants_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        grants_df.to_csv(output_filename, index=False)
        print(f"\n✓ Exported {len(grants_df)} grants to: {output_filename}")
        
        # Also create a filtered view of only currently open grants
        open_grants = grants_df[grants_df['oppStatus'] == 'posted']
        if not open_grants.empty:
            open_filename = f"open_grants_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
            open_grants.to_csv(open_filename, index=False)
            print(f"✓ Exported {len(open_grants)} currently open grants to: {open_filename}")
    
    # ========================================================================
    # Alternative: More specific example searches
    # ========================================================================
    
    print("\n\n" + "="*80)
    print("EXAMPLE: Healthcare Technology Startup")
    print("="*80)
    
    healthcare_tech_grants = api.search_business_grants(
        business_keywords=["health technology", "medical devices", "telehealth"],
        industry_focus="health",
        research_oriented=True,
        max_results=50
    )
    
    if not healthcare_tech_grants.empty:
        print(healthcare_tech_grants[['title', 'agencyName', 'closeDate']].head())
    
    print("\n" + "="*80)
    print("EXAMPLE: Clean Energy Company")
    print("="*80)
    
    energy_grants = api.search_business_grants(
        business_keywords=["renewable energy", "solar", "energy efficiency"],
        industry_focus="energy",
        research_oriented=False,
        max_results=50
    )
    
    if not energy_grants.empty:
        print(energy_grants[['title', 'agencyName', 'closeDate']].head())

BUSINESS GRANT SEARCH - Customized for Your Business
Searching SBA...
No grants found matching criteria
Searching HHS...
No grants found matching criteria
Searching DOE...
No grants found matching criteria
Searching DOD...
No grants found matching criteria
Searching NSF...
No grants found matching criteria
Searching ED...
No grants found matching criteria
Searching USDA...
No grants found matching criteria
Searching NASA...
No grants found matching criteria
Searching DHS...
No grants found matching criteria
Searching other agencies...


KeyError: 'alnist'